# Volatility Dynamics 05b — prospective test of the frozen −95m L2 motif

Claude review (**VD 010**). Module 05's −95m pattern lived only in the sparse L2 family, on 13 unique clean event timestamps, at a lag chosen after viewing the same data (discovery: L2 notch −0.097, rebound +0.093). The discovery sample cannot test it.

**Frozen specification** (`volatility_q99_motif95_prospective.py`, written to `05b/05b_frozen_spec.json` before any prospective computation):

- `DISCOVERY_END = 2026-10-08 08:00 UTC`; q99 thresholds frozen on earlier rows;
- prospective events = clean q99 onsets (no q99 bar in the previous 120 min) at/after `DISCOVERY_END`;
- primary procedure = **Module 05's own code path** (Module 04 controls on clean origins); the notch contrasts adjacent lags, so the level confound largely cancels;
- **PASS** iff ≥ 20 unique prospective timestamps **and** mean notch < 0 **and** one-sided sign-flip $p<0.05$; fewer than 20 → `INSUFFICIENT_DATA`; else `FAIL`.

Secondary (no decision): rebound > 0; notch under 04b M2 bias-corrected controls. Context only: notch at every center −115…−10 min.

Re-run this notebook as data accumulates; the specification does not change.

In [ ]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 300)
pd.set_option('display.max_rows', 300)

load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.modeling import volatility_vd_b_core as vb
from data_engineering.modeling import volatility_q99_motif95 as qm
from data_engineering.modeling import volatility_q99_motif95_prospective as mp
ASSETS = [x.strip().upper() for x in os.getenv('VD5B_ASSETS', '').split(',') if x.strip()]
SEED = int(os.getenv('VD5B_SEED', '9595'))
OUT = PROJECT_ROOT / 'notebooks' / 'Volatility_Dynamics' / '05b'
OUT.mkdir(parents=True, exist_ok=True)
print('05b', mp.VERSION, mp.selftest())
spec = {
    'version': mp.VERSION, 'discovery_end_utc': str(mp.DISCOVERY_END),
    'motif_bars': list(qm.MOTIF_BARS), 'center_bar': qm.MOTIF_CENTER_BAR,
    'l2_features': list(qm.L2_FEATURES), 'clean_lookback_min': qm.PRIOR_Q99_LOOKBACK_MIN,
    'primary_statistic': 'mean over unique prospective timestamps of l2_notch_95 (Module 05 definition)',
    'pass_rule': f'n_unique >= {mp.MIN_UNIQUE_TIMESTAMPS} and mean < 0 and one-sided sign-flip p < {mp.ALPHA}',
    'discovery_reference': {'notch': mp.DISCOVERY_NOTCH, 'rebound': mp.DISCOVERY_REBOUND, 'n_unique': mp.DISCOVERY_N_UNIQUE},
}
(OUT / '05b_frozen_spec.json').write_text(json.dumps(spec, indent=2))


In [ ]:
def qident(name):
    return '"' + str(name).replace('"', '""') + '"'

def load_frame(columns, start=None, end=None, assets=None):
    cols = list(dict.fromkeys(['state_time', 'underlying', 'return_5m'] + list(columns)))
    where = []
    if start:
        ts = pd.Timestamp(start)
        ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
        where.append(f"state_time >= TIMESTAMPTZ '{ts.isoformat()}'")
    if end:
        ts = pd.Timestamp(end)
        ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
        where.append(f"state_time < TIMESTAMPTZ '{(ts + pd.Timedelta(days=1)).isoformat()}'")
    if assets:
        where.append('underlying IN (' + ','.join(repr(x) for x in assets) + ')')
    sql = ('SELECT ' + ', '.join(qident(c) for c in cols) + ' FROM time_series_feature_state'
           + ('' if not where else ' WHERE ' + ' AND '.join(where)) + ' ORDER BY underlying, state_time')
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        df = con.execute(sql).df()
    finally:
        con.close()
    if df.empty:
        raise RuntimeError('No rows found.')
    df['state_time'] = pd.to_datetime(df['state_time'], utc=True)
    if df.duplicated(['underlying', 'state_time']).any():
        raise RuntimeError('Duplicate asset/timestamp rows.')
    print(f'rows {len(df):,} | assets {sorted(df.underlying.unique())} | {df.state_time.min()} -> {df.state_time.max()}')
    return df.reset_index(drop=True)

def schema_columns():
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        return con.execute('DESCRIBE time_series_feature_state').df()
    finally:
        con.close()

schema = set(schema_columns()['column_name'].astype(str))
l2 = [c for c in qm.L2_FEATURES if c in schema]
if len(l2) != len(qm.L2_FEATURES):
    raise RuntimeError(f'missing L2 features: {set(qm.L2_FEATURES) - set(l2)}')
df = load_frame(l2 + ['intrabar_range_bps'], None, None, ASSETS)
grid = vb.build_grid(df)


In [ ]:
events, thr = mp.frozen_events(grid)
thr_df = pd.DataFrame([{'underlying': k, 'q99_threshold_frozen': v} for k, v in thr.items()])
ref = PROJECT_ROOT / 'notebooks' / 'Volatility_Dynamics' / '05' / '05_q99_thresholds_and_counts.csv'
if ref.exists():
    chk = thr_df.merge(pd.read_csv(ref)[['underlying', 'threshold']], on='underlying', how='left')
    chk['rel_diff'] = (chk.q99_threshold_frozen / chk.threshold - 1).abs()
    print('frozen vs Module 05 thresholds, max relative diff:', float(chk.rel_diff.max()))
pros_mask = (events.event_time >= mp.DISCOVERY_END).to_numpy()
print('data end:', df.state_time.max(), '| prospective onsets (before clean filter):', int(pros_mask.sum()))


In [ ]:
res = mp.module05_procedure(grid.frame, events, pros_mask, l2)
pe = res['prospective_events']
print('clean prospective asset-events:', len(pe), '| unique timestamps:', pe.event_time.nunique() if len(pe) else 0)
rng = np.random.default_rng(SEED)
notch = mp.timestamp_series(res['components'], 'l2_notch_95')
rebound = mp.timestamp_series(res['components'], 'l2_rebound_95_to_post')
decision = {
    'primary_l2_notch': mp.decide(notch, expected_sign=-1, rng=rng),
    'secondary_l2_rebound': mp.decide(rebound, expected_sign=+1, rng=rng),
}


In [ ]:
tail = vb.tail_flags(grid, thr)
prior = vb.window_count(grid, tail, -24, -1) > 0
clean_elig = vb.eligible_origins(grid, tail) & ~prior
sec = mp.secondary_04b_notch(grid, events, set(pe.event_id) if len(pe) else set(), clean_elig, l2)
decision['secondary_l2_notch_04b_M2_bc'] = mp.decide(sec, expected_sign=-1, rng=rng)
scan = mp.center_scan(res['scores'])
print(json.dumps(decision, indent=2, default=float))
display(scan.round(3))


In [ ]:
thr_df.to_csv(OUT / '05b_frozen_thresholds.csv', index=False)
pe.to_csv(OUT / '05b_prospective_events.csv', index=False)
res['components'].to_csv(OUT / '05b_components.csv', index=False)
scan.to_csv(OUT / '05b_center_scan_context_only.csv', index=False)
sec.rename('l2_notch_04b_M2_bc').rename_axis('event_time').reset_index().to_csv(OUT / '05b_secondary_04b_notch.csv', index=False)
(OUT / '05b_decision.json').write_text(json.dumps(decision, indent=2, default=float))
meta = {
    'version': mp.VERSION, 'module': '05b_prospective_motif95', 'frozen_spec': spec,
    'data_first_time': str(df.state_time.min()), 'data_last_time': str(df.state_time.max()),
    'n_prospective_onsets_raw': int(pros_mask.sum()), 'n_clean_prospective_asset_events': int(len(pe)),
    'decision_primary': decision['primary_l2_notch']['status'],
}
(OUT / '05b_run_metadata.json').write_text(json.dumps(meta, indent=2, default=float))
print('wrote', OUT)
